In [ ]:
using LensFactory
using CairoMakie
include("FreeFormLens.jl")
using Random

In [ ]:
# Initialize default cosmology
cosmo = Cosmology.init_cosmology()

# Lens and source redshifts
zl = 0.5
zs = 1.5

# ADDs and distance ratio
Dol = Cosmology.angular_diameter_distance(cosmo, 0., zl)
Dls = Cosmology.angular_diameter_distance(cosmo, zl, zs)
Dos = Cosmology.angular_diameter_distance(cosmo, 0., zs)
adis = Dls/Dos

In [ ]:
GC.gc()
# creating a mesh
x, y = Lenses.get_meshgrid(5, 5, 0.5);
name = "050505" 
NX = size(x, 1)
NY = size(x, 2)

In [ ]:
# Initialize NFW lens
lens = Lenses.init_NSISPLens(x_c = 0., y_c = 0., v_d = 150., x_s = 1.)

fig, axes = Lenses.plot_image_plane(lens, x, y, adis)
display(fig)
# making a composite lens now.
Random.seed!(6246)

# Initialize a lens made of multiple point mass lenses
n_point = 5
ensemble = [(lens=:NSISPLens, x_c=(-3.0 + 6.0*rand()), y_c = (-3.0 + 6.0*rand()),  v_d = 150., x_s = 1.) for _ in 1:n_point]
#lens = Lenses.init_CompositeLens(ensemble)

# Get scaled deflection maps
dx, dy = Lenses.get_deflection(lens, x, y)

# Re-scale deflection maps to source plane
@. dx = adis * dx
@. dy = adis * dy

# Plot the deflection map(s)
"""fig, ax = Lenses.plot_sky(x, y)
hm = heatmap!(ax, x[:,1], y[1,:], dx, colormap=:RdBu)
display(fig)

heatmap!(ax, x[:,1], y[1,:], dy, colormap=:RdBu)
display(fig)"""

In [ ]:
# Get scaled Jacobian components
dxx, dyy, dxy = Lenses.get_jacobian(lens, x, y)
dx, dy = Lenses.get_deflection(lens, x, y) # for comparison

dx .= adis * dx
dy .= adis * dy
dx_max = maximum(abs.(dx))
dy_max = maximum(abs.(dy))
colorrange = (-max(dx_max, dy_max), max(dx_max, dy_max))

# Re-scale Jacobian components to source plane
@. dxx = adis * dxx
@. dyy = adis * dyy
@. dxy = adis * dxy

# Get convergence and shear maps from Jacobian components
kappa = 0.5 .* (dxx .+ dyy)
gamma1 = 0.5 .* (dxx .- dyy)
gamma2 = dxy

dx_, dy_ = zero(dx), zero(dy)
gridx = copy(x)
gridy = copy(y)

#FreeFormLens.deflection!(dx_, dy_, x, y, kappa ./ adis, gridx, gridy)
t = @elapsed begin
    Free_lens = FreeFormLens.init_FreeFormLens(kappa ./ adis, gridx, gridy)
end
print("Time taken to initialize FreeFormLens: ", t, " seconds\n")
dx_, dy_ = Lenses.get_deflection(Free_lens, x, y)
dx_ .*= adis
dy_ .*= adis
# Plot one of the Jacobian component map
fig, ax = Lenses.plot_sky(x, y)

hm = heatmap!(ax, x[:,1], y[1,:], dx_, colormap=:RdBu, colorrange = colorrange)
cb = Colorbar(fig[1,2], hm; label="Deflection difference (arcsec)", width=15, labelrotation=3π/2)
display(fig)

fig, ax = Lenses.plot_sky(x, y)
hm = heatmap!(ax, x[:,1], y[1,:], dx, colormap=:RdBu, colorrange = colorrange)
cb = Colorbar(fig[1,2], hm; label="Deflection difference (arcsec)", width=15, labelrotation=3π/2)
xlims!(-5,5)
ylims!(-5,5)
CairoMakie.save("αx.png", fig)

fig, ax = Lenses.plot_sky(x, y)
hm = heatmap!(ax, x[:,1], y[1,:], dy_, colormap=:RdBu, colorrange = colorrange)
cb = Colorbar(fig[1,2], hm; label="Deflection difference (arcsec)", width=15, labelrotation=3π/2)
display(fig)

fig, ax = Lenses.plot_sky(x, y)
hm = heatmap!(ax, x[:,1], y[1,:], dy, colormap=:RdBu, colorrange = colorrange)
cb = Colorbar(fig[1,2], hm; label="Deflection difference (arcsec)", width=15, labelrotation=3π/2)
xlims!(-5,5)
ylims!(-5,5)
CairoMakie.save("αy.png",fig)


In [ ]:
fig, ax = Lenses.plot_sky(x, y)
cs1 = heatmap!(ax, x[:,1], y[1,:], (dx .- dx_), colormap=:RdBu, colorrange=(-dx_max/10, dx_max/10))
cb1 = Colorbar(fig[1,2], cs1; label="αx - αx_", width=15, labelrotation=3π/2)
xlims!(ax, -5., 5.)
ylims!(ax, -5., 5.)
CairoMakie.save("αx$name.png", fig)

fig, ax = Lenses.plot_sky(x, y)
cs2 = heatmap!(ax, x[:,1], y[1,:], (dy .- dy_), colormap=:RdBu, colorrange=(-dy_max/10, dy_max/10))
cb2 = Colorbar(fig[1,2], cs2; label="αy - αy_", width=15, labelrotation=3π/2)
xlims!(ax, -5., 5.)
ylims!(ax, -5., 5.)
CairoMakie.save("αy$name.png", fig)


In [ ]:
# Get scaled Jacobian components
dxx, dyy, dxy = Lenses.get_jacobian(lens, x, y)

# Re-scale Jacobian components to source plane
@. dxx = adis * dxx
@. dyy = adis * dyy
@. dxy = adis * dxy

# Get convergence and shear maps from Jacobian components
kappa = 0.5 .* (dxx .+ dyy)
gamma1 = 0.5 .* (dxx .- dyy)
gamma2 = dxy

gamma1_max = maximum(abs.(gamma1))
gamma2_max = maximum(abs.(gamma2))
kappa_max = maximum(abs.(kappa))

# Plot one of the Jacobian component map
fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], gamma2, colormap=:RdBu, colorrange=(-gamma2_max, gamma2_max))
cb = Colorbar(fig[1,2], cs; label="γ_2", width=15, labelrotation=3π/2)
xlims!(-5,5)
ylims!(-5,5)
CairoMakie.save("γ_2.png", fig)

fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], gamma1, colormap=:RdBu, colorrange=(-gamma1_max, gamma1_max))
cb = Colorbar(fig[1,2], cs; label="γ_1", width=15, labelrotation=3π/2)
xlims!(-5,5)
ylims!(-5,5)
CairoMakie.save("γ_1.png", fig)

fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], kappa, colormap=:RdBu, colorrange=(-kappa_max, kappa_max))
cb = Colorbar(fig[1,2], cs; label="Convergence (κ)", width=15, labelrotation=3π/2)
xlims!(-5,5)
ylims!(-5,5)
CairoMakie.save("kappa.png", fig)

In [ ]:
dxx_, dyy_, dxy_ = zero(dxx), zero(dyy), zero(dxy)
gridx = copy(x)
gridy = copy(y)
FreeFormLens.jacobian!(dxx_, dyy_, dxy_, x, y, kappa ./ adis, gridx, gridy)

@. dxx_ *= adis
@. dyy_ *= adis
@. dxy_ *= adis
kappa_ = 0.5 .* (dxx_ .+ dyy_)
gamma2_ = dxy_
gamma1_ = 0.5 .* (dxx_ .- dyy_)

In [ ]:
# Plot one of the Jacobian component map
"""fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], gamma2_, colormap=:RdBu, colorrange=(-gamma2_max, gamma2_max))
cb = Colorbar(fig[1,2], cs; label="γ_2", width=15, labelrotation=3π/2)
display(fig)

fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], gamma1_, colormap=:RdBu, colorrange=(-gamma1_max, gamma1_max))
cb = Colorbar(fig[1,2], cs; label="γ_1", width=15, labelrotation=3π/2)
display(fig)

fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], kappa_, colormap=:RdBu, colorrange=(-kappa_max, kappa_max))
cb = Colorbar(fig[1,2], cs; label="Convergence (κ)", width=15, labelrotation=3π/2)
display(fig)"""

In [ ]:
# Plot differences
fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], (gamma2_.- gamma2), colormap=:RdBu, colorrange=(-gamma2_max/10, gamma2_max/10))
cb = Colorbar(fig[1,2], cs; label="γ_2", width=15, labelrotation=3π/2)
xlims!(ax, -5., 5.)
ylims!(ax, -5., 5.)
CairoMakie.save("γ2_diff$name.png", fig)

fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], (gamma1_.- gamma1), colormap=:RdBu, colorrange=(-gamma1_max/10, gamma1_max/10))
cb = Colorbar(fig[1,2], cs; label="γ_1", width=15, labelrotation=3π/2)
xlims!(ax, -5., 5.)
ylims!(ax, -5., 5.)
CairoMakie.save("γ1_diff$name.png", fig)

fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], (kappa_.- kappa), colormap=:RdBu, colorrange=(-kappa_max/10, kappa_max/10))
cb = Colorbar(fig[1,2], cs; label="Convergence (κ)", width=15, labelrotation=3π/2)
xlims!(ax, -5., 5.)
ylims!(ax, -5., 5.)
CairoMakie.save("kappa_diff$name.png", fig)

In [ ]:
# trying with a completely different kappa — uniform kappa = 1
kappa_test = ones(size(x))

dxx_test = zero(x)
dyy_test = zero(x)
dxy_test = zero(x)

FreeFormLens.jacobian!(dxx_test, dyy_test, dxy_test, x, y, kappa_test, x, y)

gamma2_test = dxy_test
gamma1_test = 0.5 .* (dxx_test .- dyy_test)
kappa_test = 0.5 .* (dxx_test .+ dyy_test)

fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], gamma2_test, colormap=:RdBu)
cb = Colorbar(fig[1,2], cs; label="γ_2", width=15, labelrotation=3π/2)
xlims!(ax, -5., 5.)
ylims!(ax, -5., 5.)
CairoMakie.save("γ2_flatsheet$name.png", fig)

fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], gamma1_test, colormap=:RdBu)
cb = Colorbar(fig[1,2], cs; label="γ_1", width=15, labelrotation=3π/2)
xlims!(ax, -5., 5.)
ylims!(ax, -5., 5.)
CairoMakie.save("γ1_flatsheet$name.png", fig)

fig, ax = Lenses.plot_sky(x, y)
cs = heatmap!(ax, x[:,1], y[1,:], kappa_test, colormap=:RdBu)
cb = Colorbar(fig[1,2], cs; label="Convergence (κ)", width=15, labelrotation=3π/2)
xlims!(ax, -5., 5.)
ylims!(ax, -5., 5.)
CairoMakie.save("kappa_flatsheet$name.png", fig)